# graphArt_LM: Phase 1 prompting on a free Kaggle GPU

Runs the six prompts over the same 40 stratified test samples as every other run, on a
local open model served by Ollama. Only inference happens here; scoring happens back on
the laptop with the unchanged `pipeline.score`.

**Before running**, in the right-hand panel under *Session options*:
- **Accelerator:** GPU T4 x2
- **Internet:** on (needed to install Ollama and download the model)

**To run unattended**, use *Save Version → Save & Run All (Commit)*. It runs in the
background with the browser closed, for up to 12 hours, and keeps everything written to
`/kaggle/working`.

**When it finishes**, download `graphart_results.zip` from the version's *Output* tab and
unzip it into the repo root. It contains `results/raw/ollama__<model>__<prompt>/`. Then
locally, for each model and prompt:

```
python -m pipeline.run_prompting --backend ollama --model <model> --prompt <prompt> --stratify 10 --cached-only
python -m pipeline.score results/ollama__<model>__<prompt>.jsonl
```

Generated by `scripts/make_kaggle_notebook.py`. Edit that script, not this file.

In [ ]:
# MODELS       one per session is the safe default. qwen2.5-coder:14b fits one T4 and
#              should take ~3 h for all six prompts; "gemma4:31b" needs both T4s and
#              roughly 6-9 h, and a session is capped at 12.
# PROMPTS      highest value first, so a session that runs out of time loses the least.
# MAX_TOKENS   cot and cos write their working before the graph; the runner's default
#              cap (1500 tokens) would cut them off mid-thought.
# TIME_BUDGET  stop starting new work with an hour of the 12 to spare.
# SMOKE_ONLY   True: install, download, try one sample, and stop.
# VALIDATE_ONLY True: only check the code arrived intact. Run it on a CPU session
#              (Accelerator: None), which costs no GPU quota.
MODELS = ["qwen2.5-coder:7b"]; PROMPTS = ["zero_shot", "few_shot"]; MAX_TOKENS = {"cot": 4096, "cos": 4096}; TIME_BUDGET_HOURS = 11.0; SMOKE_ONLY = False; VALIDATE_ONLY = False

## 1. Unpack the code, prompts and samples

In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

START = time.time()
WORK = Path(os.environ.get("GRAPHART_WORK", "/kaggle/working"))
REPO = WORK / "graphArt_LM"
BUNDLE_SHA256 = "cb130b286ae91e8f7f5929db70a219ee4ee8677bc3161dae7cab32d5638c99b4"
BUNDLE = (
    "..."
    "n/1n/9l/9p9/t8//B91wpEsA4AEA"
)
data = base64.b64decode(BUNDLE)
assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256, "bundle damaged -- regenerate the notebook"
REPO.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as tar:
    tar.extractall(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(sorted(str(p.relative_to(REPO)) for p in REPO.rglob("*") if p.is_file()))
if VALIDATE_ONLY:
    raise SystemExit("bundle intact -- VALIDATE_ONLY is set, so stopping here")
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

## 2. Install and start Ollama

In [ ]:
def ollama_up() -> bool:
    try:
        print(urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2).read().decode())
        return True
    except OSError:
        return False

if not shutil.which("ollama"):
    subprocess.run("apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null",
                   shell=True, check=True)
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

if not ollama_up():
    # Ollama splits a model over both T4s by itself when it does not fit on one.
    env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="24h")
    subprocess.Popen(["ollama", "serve"], env=env,
                     stdout=open(WORK / "ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(1)
        if ollama_up():
            break
    else:
        raise RuntimeError(f"ollama did not start -- see {WORK / 'ollama.log'}")

for model in MODELS:
    subprocess.run(["ollama", "pull", model], check=True)
subprocess.run(["ollama", "list"])

## 3. Smoke test: one sample, before committing hours to it

In [ ]:
# Shows the reply, how long it took and how long it is, so a model that thinks out loud
# or ignores the format is obvious before the full run.
from pipeline.backends import make_backend
from pipeline.run_prompting import load_prompt, load_split, _render

row = load_split("test", None, None, 10)[0]
system, template = load_prompt("zero_shot")
for model in MODELS:
    t = time.time()
    reply = make_backend("ollama", model)(system, _render(template, row))
    took = time.time() - t
    print(f"=== {model}: {took:.0f}s, {len(reply)} chars\n{reply[:1500]}")
    # GPU hours are scarce: a model that does not answer in the format would burn the
    # whole session producing nothing scorable, so stop here instead.
    if "NODES" not in reply or "EDGES" not in reply:
        raise SystemExit(f"{model}: the smoke-test reply is not a graph -- stopping before the full run")
    # cot and cos write roughly 4x as much, so a zero_shot sample is a lower bound.
    print(f"rough projection for {model}: {took * 40 * (len(PROMPTS) + 3 * sum(p in ('cot', 'cos') for p in PROMPTS)) / 3600:.1f} h")

## 4. The run

In [ ]:
def remaining() -> float:
    return TIME_BUDGET_HOURS * 3600 - (time.time() - START)

def package() -> None:
    # Re-zipped after every prompt, so a session that dies late keeps what it finished.
    if (REPO / "results").exists():
        shutil.make_archive(str(WORK / "graphart_results"), "zip", REPO, "results")

for model in ([] if SMOKE_ONLY else MODELS):
    for prompt in PROMPTS:
        if remaining() < 600:
            print(f"time budget spent -- skipping {model} / {prompt}")
            continue
        cmd = [sys.executable, "-m", "pipeline.run_prompting", "--backend", "ollama",
               "--model", model, "--prompt", prompt, "--stratify", "10"]
        if prompt in MAX_TOKENS:
            cmd += ["--max-tokens", str(MAX_TOKENS[prompt])]
        print(f"\n##### {model} / {prompt}   ({remaining() / 3600:.1f} h left)", flush=True)
        try:
            # Cached replies survive a timeout; a later session would only redo the rest.
            subprocess.run(cmd, check=False, timeout=remaining())
        except subprocess.TimeoutExpired:
            print("stopped by the time budget")
        package()

package()
print(f"\ndone in {(time.time() - START) / 3600:.1f} h")
raw_dir = REPO / "results" / "raw"
for run in sorted(raw_dir.iterdir()) if raw_dir.exists() else []:
    print(f"{run.name:55s} {len(list(run.glob('*.txt'))):3d} replies")
for meta in sorted((REPO / "results").glob("*.meta.json")):
    m = json.loads(meta.read_text())
    print(f"{meta.stem:55s} n={m['n']} failed={m['failed']}")
if not SMOKE_ONLY:
    print(f"download {WORK / 'graphart_results.zip'} from the Output tab")